In [ ]:
import os
import glob
import cv2
import random
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import pandas as pd
from collections import defaultdict

# --- CONFIGURACIÓN ---
dataset_path = '../data/dataset'
splits = ['train', 'valid', 'test']
class_names = ['Bee', 'Asian Hornet']
target_size = (640, 640)

In [ ]:
# --- ANÁLISIS ESTADÍSTICO ---
def analizar_dataset(base_path, splits):
    print(f"Analizando dataset en: {base_path}\n" + "-"*40)

    total_images = 0
    image_paths = []

    for split in splits:
        # Construir ruta: dataset/train/images
        path_imgs = os.path.join(base_path, split, 'images')

        if not os.path.exists(path_imgs):
            print(f"⚠️ Aviso: No se encontró la carpeta {split} en {path_imgs}")
            continue

        # Buscar extensiones comunes
        files = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
            files.extend(glob.glob(os.path.join(path_imgs, ext)))

        count = len(files)
        total_images += count
        print(f"📁 {split.upper()}: {count} imágenes encontradas.")

        if count > 0:
            image_paths.extend(files)
            sample_img = cv2.imread(files[0])
            if sample_img is not None:
                h, w, c = sample_img.shape
                print(f"   Formato típico: {files[0].split('.')[-1]}")
                print(f"   Tamaño de muestra: {w}x{h} pixeles, {c} canales")

    print("-" * 40)
    print(f" TOTAL DE IMÁGENES: {total_images}")
    return image_paths

# --- VISUALIZAR EJEMPLOS CON CAJAS ---
def visualizar_ejemplos(img_paths, num_samples=3):
    if len(img_paths) < num_samples:
        print("No hay suficientes imágenes para mostrar ejemplos.")
        return

    # Seleccionar imágenes aleatorias
    samples = random.sample(img_paths, num_samples)

    plt.figure(figsize=(15, 5))

    for i, img_path in enumerate(samples):
        # Cargar imagen
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convertir a RGB para matplotlib
        h_img, w_img, _ = img.shape

        # Inferir ruta del label
        # Asume estructura: .../images/foto.jpg -> .../labels/foto.txt
        label_path = img_path.replace('images', 'labels').rsplit('.', 1)[0] + '.txt'

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split()
                class_id = int(parts[0])

                # Coordenadas YOLO normalizadas (0 a 1)
                x_center, y_center, w, h = map(float, parts[1:5])

                # Convertir a píxeles
                x1 = int((x_center - w/2) * w_img)
                y1 = int((y_center - h/2) * h_img)
                x2 = int((x_center + w/2) * w_img)
                y2 = int((y_center + h/2) * h_img)

                # Dibujar caja
                color = (0, 255, 0) # Verde
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

                # Poner texto (Clase)
                label_text = class_names[class_id] if class_id < len(class_names) else f"ID: {class_id}"
                cv2.putText(img, label_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        else:
            print(f"⚠️ Label no encontrado para: {os.path.basename(img_path)}")

        # Mostrar en subplot
        plt.subplot(1, num_samples, i+1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Ejemplo {i+1}")

    plt.tight_layout()
    plt.show()



In [ ]:
# --- EJECUCIÓN ---
path_list = analizar_dataset(dataset_path, splits)
visualizar_ejemplos(path_list, 3)

In [ ]:
def normalizar_imagenes(base_path, splits, size):
    print(f"Iniciando normalización a {size} en: {base_path}")

    resized_count = 0
    converted_count = 0
    total_processed = 0

    for split in splits:
        path_imgs = os.path.join(base_path, split, 'images')

        # Obtener lista de imágenes
        files = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
            files.extend(glob.glob(os.path.join(path_imgs, ext)))

        print(f"\n Procesando carpeta: {split} ({len(files)} imágenes)")

        for img_path in tqdm(files):
            img = cv2.imread(img_path)

            if img is None:
                print(f"Error leyendo: {img_path}")
                continue

            h, w = img.shape[:2]

            # Chequeo de Canales (Grayscale vs Color)
            needs_save = False
            if len(img.shape) == 2:
                img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
                converted_count += 1
                needs_save = True
            elif img.shape[2] != 3:
                # Caso raro: imagen con 4 canales (PNG con transparencia)
                img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
                converted_count += 1
                needs_save = True

            # Chequeo de Dimensiones
            if (w, h) != size:
                img = cv2.resize(img, size, interpolation=cv2.INTER_LINEAR)
                resized_count += 1
                needs_save = True

            # Sobrescribir solo si hubo cambios para ahorrar
            if needs_save:
                cv2.imwrite(img_path, img)

            total_processed += 1

    print("-" * 40)
    print("Proceso finalizado.")
    print(f"Imágenes procesadas: {total_processed}")
    print(f"Imágenes redimensionadas: {resized_count}")
    print(f"Imágenes convertidas a 3 canales: {converted_count}")



In [ ]:
# --- EJECUTAR ---
normalizar_imagenes(dataset_path, splits, target_size)

In [ ]:
def contar_distribucion_yolo(base_path, splits, class_mapping=None):
    print(f"Analizando distribución de clases en: {base_path}\n")

    global_counts = defaultdict(int)
    global_negatives = 0
    data_summary = []

    for split in splits:
        # Ruta a las etiquetas: dataset/train/labels
        path_labels = os.path.join(base_path, split, 'labels')

        if not os.path.exists(path_labels):
            print(f"⚠️ No se encontró la carpeta labels en: {split}")
            continue

        txt_files = glob.glob(os.path.join(path_labels, '*.txt'))

        split_counts = defaultdict(int)
        split_negatives = 0

        for txt_file in txt_files:
            with open(txt_file, 'r') as f:
                lines = f.readlines()

                # Si el archivo está vacío, es una imagen negativa (background)
                if not lines:
                    split_negatives += 1
                else:
                    # Recorrer cada línea (cada objeto)
                    for line in lines:
                        parts = line.strip().split()
                        if parts:
                            class_id = int(parts[0])
                            split_counts[class_id] += 1
                            global_counts[class_id] += 1

        # Guardar datos para mostrar luego
        global_negatives += split_negatives
        print(f"SPLIT: {split.upper()}")
        print(f"   Imágenes Negativas (vacías): {split_negatives}")
        print(f"   Conteo por clase (instancias):")

        sorted_ids = sorted(split_counts.keys())
        for cid in sorted_ids:
            name = class_mapping[cid] if class_mapping and cid < len(class_mapping) else f"Class {cid}"
            count = split_counts[cid]
            print(f"      • {name}: {count}")

            # Guardar para tabla
            data_summary.append({
                'Split': split,
                'Clase': name,
                'Instancias': count
            })
        print("-" * 30)

    # --- RESUMEN FINAL ---
    print(f"\nTOTAL GLOBAL DEL DATASET")
    print(f"   Total Imágenes Negativas: {global_negatives}")

    # Crear DataFrame
    if data_summary:
        df = pd.DataFrame(data_summary)

        # Visualización gráfica
        plt.figure(figsize=(12, 6))

        # Agrupar por clase para el gráfico total
        total_per_class = df.groupby('Clase')['Instancias'].sum().sort_values(ascending=False)

        total_per_class.plot(kind='bar', color='skyblue', edgecolor='black')
        plt.title('Distribución Total de Clases (Instancias)')
        plt.ylabel('Cantidad de etiquetas')
        plt.xlabel('Clases')
        plt.xticks(rotation=45)
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()

        return df
    else:
        print("No se encontraron etiquetas.")
        return None



In [ ]:
# --- EJECUTAR ---
df_resultados = contar_distribucion_yolo("../data/dataset_mini", splits, class_names)

In [ ]:
import os
import shutil
import random
from pathlib import Path
from tqdm import tqdm

def crear_dataset_pequeno(origen, destino, limites):
    origen = Path(origen)
    destino = Path(destino)

    # 1. Crear estructura de carpetas en el destino
    if destino.exists():
        print(f"¡Atención! La carpeta '{destino}' ya existe. Borrándola para empezar de cero...")
        shutil.rmtree(destino)

    destino.mkdir(parents=True)
    print(f"Creado directorio: {destino}")

    # 2. Copiar el archivo data.yaml
    yaml_file = origen / "data.yaml"
    if yaml_file.exists():
        shutil.copy(yaml_file, destino / "data.yaml")
        print("Copiado data.yaml")
    else:
        print("ADVERTENCIA: No se encontró data.yaml en el origen.")

    # Extensiones de imagen soportadas
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

    # 3. Procesar cada split (train, valid, test)
    for split, cantidad_deseada in limites.items():
        print(f"\nProcesando '{split}'...")

        # Rutas de origen
        split_path_origen = origen / split
        # Intentar detectar si usa subcarpetas 'images'/'labels' o estructura plana
        if (split_path_origen / "images").exists():
            img_dir_origen = split_path_origen / "images"
            lbl_dir_origen = split_path_origen / "labels"
            usa_subcarpetas = True
        else:
            img_dir_origen = split_path_origen
            lbl_dir_origen = split_path_origen
            usa_subcarpetas = False

        if not split_path_origen.exists():
            print(f"  Saltando: No existe la carpeta '{split}' en el origen.")
            continue

        # Recolectar todas las imágenes
        todas_imagenes = [
            f for f in img_dir_origen.iterdir()
            if f.suffix.lower() in valid_extensions and f.is_file()
        ]

        total_encontrado = len(todas_imagenes)
        print(f"  Encontradas {total_encontrado} imágenes.")

        # Determinar cuántas copiar
        if total_encontrado < cantidad_deseada:
            print(f"  Advertencia: Solo hay {total_encontrado}, se copiarán todas (se pedían {cantidad_deseada}).")
            seleccion = todas_imagenes
        else:
            seleccion = random.sample(todas_imagenes, cantidad_deseada)

        # Preparar directorios de destino
        split_path_dest = destino / split
        if usa_subcarpetas:
            (split_path_dest / "images").mkdir(parents=True, exist_ok=True)
            (split_path_dest / "labels").mkdir(parents=True, exist_ok=True)
        else:
            split_path_dest.mkdir(parents=True, exist_ok=True)

        # Copiar archivos
        copiados = 0
        for img_path in tqdm(seleccion, desc=f"  Copiando {split}", unit="img"):
            # Definir rutas destino
            if usa_subcarpetas:
                dest_img = split_path_dest / "images" / img_path.name
            else:
                dest_img = split_path_dest / img_path.name

            # Copiar Imagen
            shutil.copy(img_path, dest_img)

            # Buscar y Copiar Etiqueta (txt)
            label_name = img_path.stem + ".txt"
            label_path = lbl_dir_origen / label_name

            if label_path.exists():
                if usa_subcarpetas:
                    dest_lbl = split_path_dest / "labels" / label_name
                else:
                    dest_lbl = split_path_dest / label_name
                shutil.copy(label_path, dest_lbl)

            copiados += 1

        print(f"  Terminado: {copiados} imágenes copiadas a '{split}'.")

# --- CONFIGURACIÓN ---
DIRECTORIO_ORIGEN = "../data/dataset"
DIRECTORIO_NUEVO = "../data/dataset_mini"

# Cantidades que pediste
CONTEOS = {
    "train": 6000,
    "valid": 200,
    "test": 400
}

if __name__ == "__main__":
    crear_dataset_pequeno(DIRECTORIO_ORIGEN, DIRECTORIO_NUEVO, CONTEOS)